### Benchmark formal + HF Inference Endpoints

Dos cosas que faltaban del modulo 4:

1. **Benchmark formal**: en vez de mirar un output y opinar si "esta bien", le damos al modelo una bateria de problemas chicos con tests automaticos (`assert`) y contamos cuantos resuelve. Version mini de lo que hace HumanEval en la industria (164 problemas, metrica pass@k).
2. **HF Inference Endpoints**: en vez de bajar el modelo entero (como Whisper/Llama en modulo 3), lo llamamos por API -- corre en el servidor de HuggingFace, no en nuestra GPU.

#### Los problemas del benchmark

Cada problema define: enunciado, firma de la funcion pedida, y un set de `assert` que la solucion tiene que pasar. El nombre de la funcion en el codigo generado tiene que coincidir para poder testearla.

In [1]:
problemas = [
    {
        "nombre": "es_palindromo",
        "enunciado": "Escribi una funcion Python `es_palindromo(s: str) -> bool` que devuelva True si el string es igual leido al derecho y al reves (ignorar mayusculas/minusculas).",
        "tests": [
            ("reconocer", "es_palindromo('reconocer')", True),
            ("Ana", "es_palindromo('Ana')", True),
            ("hola", "es_palindromo('hola')", False),
        ],
    },
    {
        "nombre": "fibonacci",
        "enunciado": "Escribi una funcion Python `fibonacci(n: int) -> int` que devuelva el n-esimo numero de Fibonacci (fibonacci(0)=0, fibonacci(1)=1).",
        "tests": [
            ("fibonacci(0)", "fibonacci(0)", 0),
            ("fibonacci(1)", "fibonacci(1)", 1),
            ("fibonacci(10)", "fibonacci(10)", 55),
        ],
    },
    {
        "nombre": "invertir_palabras",
        "enunciado": "Escribi una funcion Python `invertir_palabras(s: str) -> str` que invierta el orden de las palabras en un string, separadas por espacio.",
        "tests": [
            ("hola mundo", "invertir_palabras('hola mundo')", "mundo hola"),
            ("uno dos tres", "invertir_palabras('uno dos tres')", "tres dos uno"),
        ],
    },
    {
        "nombre": "suma_digitos",
        "enunciado": "Escribi una funcion Python `suma_digitos(n: int) -> int` que sume los digitos de un numero entero positivo.",
        "tests": [
            ("suma_digitos(1234)", "suma_digitos(1234)", 10),
            ("suma_digitos(9)", "suma_digitos(9)", 9),
        ],
    },
]
print(f"{len(problemas)} problemas cargados")

4 problemas cargados


#### Helper: generar solucion, extraer codigo, correr tests

`exec()` corre el codigo generado en un namespace aislado y despues llamamos cada test contra ese namespace. Riesgo real de `exec` sobre codigo no confiable en produccion -- aca lo aceptamos porque es un ejercicio controlado (nosotros armamos los prompts, sabemos que no le estamos pidiendo nada malicioso).

In [2]:
import re

def extraer_codigo(texto):
    match = re.search(r"```(?:python)?\n(.*?)```", texto, re.DOTALL)
    return match.group(1) if match else texto

def correr_tests(codigo, problema):
    namespace = {}
    try:
        exec(codigo, namespace)
    except Exception as e:
        return False, f"error al definir la funcion: {e}"

    for nombre_test, expresion, esperado in problema["tests"]:
        try:
            resultado = eval(expresion, namespace)
        except Exception as e:
            return False, f"test '{nombre_test}' exploto: {e}"
        if resultado != esperado:
            return False, f"test '{nombre_test}' fallo: esperado {esperado!r}, obtuvo {resultado!r}"
    return True, "todos los tests pasaron"

#### Correr el benchmark contra dos modelos

- **Gemini** (frontier, via API que ya venimos usando).
- **Llama-3.1-8B-Instruct** (modelo abierto, via HF Inference Endpoints -- lo llamamos por API, no lo bajamos). Nota: `Qwen2.5-Coder-32B` (mas especializado en codigo) agoto la cuota gratis del proveedor que lo sirve durante las pruebas -- cada modelo en el router de HF pasa por un proveedor distinto (Novita, Together, etc.) con cuota propia, asi que cambiar de modelo esquiva el limite.

In [3]:
from dotenv import load_dotenv
from google import genai
from huggingface_hub import InferenceClient
import os

load_dotenv()
gemini = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
hf = InferenceClient(token=os.getenv("HF_TOKEN"))

def resolver_con_gemini(enunciado):
    prompt = f"{enunciado}\nDevolveme SOLO el codigo Python de la funcion, sin explicacion."
    r = gemini.models.generate_content(model="gemini-flash-lite-latest", contents=prompt)
    return extraer_codigo(r.text)

def resolver_con_llama(enunciado):
    prompt = f"{enunciado}\nDevolveme SOLO el codigo Python de la funcion, sin explicacion."
    r = hf.chat_completion(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
    )
    return extraer_codigo(r.choices[0].message.content)

In [4]:
resultados = {"Gemini": [], "Llama-3.1-8B (HF Endpoint)": []}

for problema in problemas:
    codigo_gemini = resolver_con_gemini(problema["enunciado"])
    ok_gemini, detalle_gemini = correr_tests(codigo_gemini, problema)
    resultados["Gemini"].append(ok_gemini)
    print(f"[Gemini] {problema['nombre']}: {'PASS' if ok_gemini else 'FAIL'} -- {detalle_gemini}")

    codigo_llama = resolver_con_llama(problema["enunciado"])
    ok_llama, detalle_llama = correr_tests(codigo_llama, problema)
    resultados["Llama-3.1-8B (HF Endpoint)"].append(ok_llama)
    print(f"[Llama]  {problema['nombre']}: {'PASS' if ok_llama else 'FAIL'} -- {detalle_llama}")
    print()

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[Gemini] es_palindromo: PASS -- todos los tests pasaron


[Llama]  es_palindromo: PASS -- todos los tests pasaron



[Gemini] fibonacci: PASS -- todos los tests pasaron


[Llama]  fibonacci: PASS -- todos los tests pasaron



[Gemini] invertir_palabras: PASS -- todos los tests pasaron


[Llama]  invertir_palabras: PASS -- todos los tests pasaron



[Gemini] suma_digitos: PASS -- todos los tests pasaron


[Llama]  suma_digitos: PASS -- todos los tests pasaron



In [5]:
print("--- pass@1 (aproximado, 1 intento por problema) ---")
for modelo, oks in resultados.items():
    tasa = sum(oks) / len(oks)
    print(f"{modelo}: {sum(oks)}/{len(oks)} ({tasa:.0%})")

--- pass@1 (aproximado, 1 intento por problema) ---
Gemini: 4/4 (100%)
Llama-3.1-8B (HF Endpoint): 4/4 (100%)
